# Lesson 7: ToolNode for Arithmetic Tools

## Objective
Learn how to use LangGraph's ToolNode to give LLM access to arithmetic functions via function calling.

## Problem Statement
**Previous lesson (Lesson 6):** We used the LLM to interpret questions, but we had to manually parse text output to extract the operation and numbers.

**Current problem:** 
- Parsing LLM text is fragile and error-prone
- What if the LLM says "eight" instead of "8"?
- How do we let the LLM call functions directly instead of outputting text?

**Solution:** Use **ToolNode** to bind functions to the LLM and let it call them directly.

## Theory Explanation

### What is a Tool in LangGraph?
A **Tool** is a function that the LLM can call. Tools are defined using Python functions with type hints and docstrings.

### How does Tool Calling Work?
1. Define function with `@tool` decorator
2. LLM sees function signature and docstring
3. User asks a question
4. LLM decides which tool to call and with what arguments
5. LangGraph executes the tool
6. Result is passed back to LLM

### What is ToolNode?
ToolNode is a graph node that:
- Takes LLM tool calls from messages
- Executes the called tools
- Returns results back to graph

### Why ToolNode vs Parsing Text?
| Approach | Pros | Cons |
|----------|------|------|
| Text parsing (Lesson 6) | Simple | Fragile, error-prone, hard to scale |
| ToolNode (Lesson 7) | Structured, reliable, LLM-native | Requires more setup |

## What is New in This Lesson?

**In Lesson 6:** LLM outputs text → we parse it manually

**In Lesson 7:**
- Define arithmetic tools with `@tool` decorator
- Bind tools to LLM with `llm.bind_tools(tools)`
- Add ToolNode to graph
- Use `tools_condition` to route to ToolNode
- LLM outputs structured tool calls

## Which Imports / APIs Solve This Problem?

```python
from langchain_core.tools import tool           # @tool decorator
from langgraph.prebuilt import ToolNode         # Execute tools
from langgraph.prebuilt import tools_condition  # Route to tools
```

**Key lines:**
- `@tool` → Mark function as executable by LLM
- `llm.bind_tools(tools)` → Give LLM access to tools
- `ToolNode(tools)` → Graph node that executes tools
- `tools_condition` → Router that sends messages to ToolNode when tools are called

## Production Insight

In production:
1. **Tool Calling is Cheaper**: LLM outputs a function call (tokens) instead of rambling (more tokens)
2. **Structured Output**: Tools define exact input/output types - no guessing
3. **Error Handling**: We can catch tool exceptions and ask LLM to retry
4. **Auditability**: Every tool call is logged and reproducible
5. **Security**: Tools are whitelist of allowed functions - LLM can't run arbitrary code

## Full Working Code

### Setup: Dependencies

In [ ]:
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool  # NEW: @tool decorator

load_dotenv()
vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)

llm = ChatVertexAI(
    model="gemini-2.5-flash",
    temperature=0
)
print("✓ Initialized LLM")

### Step 1: Define Arithmetic Tools

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """
    Add two numbers.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Sum of a and b
    """
    result = a + b
    print(f"  → Tool call: add({a}, {b}) = {result}")
    return result

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Product of a and b
    """
    result = a * b
    print(f"  → Tool call: multiply({a}, {b}) = {result}")
    return result

@tool
def divide(a: int, b: int) -> int:
    """
    Divide two numbers (integer division).
    
    Args:
        a: Numerator
        b: Denominator
    
    Returns:
        Integer quotient of a / b
    """
    if b == 0:
        return "Error: Division by zero"
    result = a // b
    print(f"  → Tool call: divide({a}, {b}) = {result}")
    return result

# Collect tools in a list
tools = [add, multiply, divide]
print("✓ Defined tools: add, multiply, divide")

### Step 2: Bind Tools to LLM

In [ ]:
# Bind tools to LLM - now LLM can see and call these functions
llm_with_tools = llm.bind_tools(tools)
print(f"✓ Bound {len(tools)} tools to LLM")
print(f"  Tools: {[tool.name for tool in tools]}")

### Step 3: Define State Schema

In [ ]:
from typing import TypedDict, List

class ArithmeticState(TypedDict):
    """State for arithmetic with tool calling"""
    question: str               # User's arithmetic question
    messages: List              # Conversation with LLM
    result: int                 # Final computed result
    tool_calls_count: int      # Track how many tool calls made

print("✓ Defined ArithmeticState")

### Step 4: Create LLM Node (Improved)

def llm_node(state: ArithmeticState) -> ArithmeticState:
    """
    Call LLM to interpret question and decide which tool to call.
    
    Unlike Lesson 6, LLM will output tool calls, not text.
    """
    messages = list(state["messages"])
    
    # First message: add the user question
    if not messages:
        messages.append(HumanMessage(content=state["question"]))
        print(f"  → Added user question: {state['question']}")
    
    # Call LLM with tools bound
    print(f"  → Calling LLM (with tools bound)...")
    response = llm_with_tools.invoke(messages)  # Note: llm_with_tools, not llm
    
    # Add response to messages
    messages.append(response)
    print(f"  → LLM response has {len(response.tool_calls) if hasattr(response, 'tool_calls') else 0} tool call(s)")
    
    return {
        "question": state["question"],
        "messages": messages,
        "result": 0,
        "tool_calls_count": 0
    }

print("✓ Defined llm_node")

### Step 5: Create ToolNode for Executing Tools

In [ ]:
from langgraph.prebuilt import ToolNode

# NEW: Create a ToolNode that executes our arithmetic tools
tool_node = ToolNode(tools)
print("✓ Created ToolNode for executing tools")

def execute_tools(state: ArithmeticState) -> ArithmeticState:
    """
    Wrapper to execute tools and update state.
    """
    # Get current messages
    messages = list(state["messages"])
    
    # Tool node returns updated messages with tool results
    result_messages = tool_node.invoke({"messages": messages})["messages"]
    
    # Extract the tool result (last message should be ToolMessage)
    last_message = result_messages[-1]
    result = last_message.content  # This is the numeric result
    
    print(f"  → Tool executed, result: {result}")
    
    return {
        "question": state["question"],
        "messages": result_messages,
        "result": result,
        "tool_calls_count": state["tool_calls_count"] + 1
    }

print("✓ Defined execute_tools wrapper")

### Step 6: Create Router Function

In [ ]:
def router(state: ArithmeticState) -> str:
    """
    Route to tools or end based on whether LLM called a tool.
    
    If LLM response has tool calls → go to "tools"
    If LLM response is just text → go to "END"
    """
    messages = state["messages"]
    last_message = messages[-1]
    
    # Check if last message has tool calls
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"  → Router: Found tool calls, routing to 'tools'")
        return "tools"
    else:
        print(f"  → Router: No tool calls, routing to 'END'")
        return "__end__"

print("✓ Defined router function")

### Step 7: Build the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

# Create graph
graph = StateGraph(ArithmeticState)

# Add nodes
graph.add_node("llm_node", llm_node)
graph.add_node("tools", execute_tools)  # NEW: Tool execution node

# Add edges
graph.add_edge(START, "llm_node")
graph.add_conditional_edges(  # NEW: Conditional routing
    "llm_node",
    router,
    {"tools": "tools", "__end__": END}
)
graph.add_edge("tools", END)

# Compile
arithmetic_graph = graph.compile()
print("✓ Compiled graph")

### Step 8: Visualize the Graph

from IPython.display import Image, display

graph_image = arithmetic_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))
print("✓ Graph visualization complete")

### Step 9: Run the Graph

# Test with different arithmetic questions
test_questions = [
    "What is 5 plus 3?",
    "Multiply 4 and 7",
    "What is 10 divided by 2?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n=== Test {i}: {question} ===")
    
    initial_state = {
        "question": question,
        "messages": [],
        "result": 0,
        "tool_calls_count": 0
    }
    
    final_state = arithmetic_graph.invoke(initial_state)
    
    print(f"\n→ Result: {final_state['result']}")
    print(f"→ Tool calls made: {final_state['tool_calls_count']}")

## Step-by-Step Code Explanation

### 1. **Using @tool Decorator**
```python
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b
```
**Why?**
- `@tool` marks function as callable by LLM
- Type hints tell LLM what arguments to pass
- Docstring explains what the function does to the LLM
- LLM automatically learns the function signature

### 2. **Binding Tools to LLM**
```python
llm_with_tools = llm.bind_tools([add, multiply, divide])
```
**Why?**
- `bind_tools()` tells LLM about available functions
- Gemini will output function calls instead of plain text
- Creates a new model instance with tools embedded

### 3. **Router Function**
```python
def router(state):
    if last_message.tool_calls:
        return "tools"  # Go to tool execution
    else:
        return "__end__"  # Done
```
**Why?**
- Decides whether LLM called a tool or finished
- `\_\_end\_\_` is magic string meaning "go to END node"
- Allows conditional graph flow based on LLM decision

### 4. **execute_tools Wrapper**
```python
result_messages = tool_node.invoke({"messages": messages})
```
**Why?**
- ToolNode executes the tool and creates ToolMessage
- ToolMessage contains the tool result (numeric output)
- We extract result and update state

## Summary

**What changed from Lesson 6?**
1. Lesson 6: LLM outputs text, we parse it
2. Lesson 7: LLM outputs tool calls, tools execute automatically

**Why is this better?**
- **Structured**: No more fragile text parsing
- **Deterministic**: Tool always returns a number, not "eight" or "8.0"
- **Extensible**: Add new tools without changing routing logic
- **Auditable**: Every tool call is logged with exact arguments

## Why This Matters in Production

1. **Tool Calling is an LLM Native Pattern**: Modern LLMs (Gemini, GPT-4, Claude) are trained on tool calling
2. **Reliability**: Tools enforce input/output types - no invalid data
3. **Scalability**: Can bind 100+ tools without changing graph logic
4. **Error Recovery**: Can catch tool exceptions and ask LLM to retry
5. **Cost**: Tool calls are cheaper than generating full sentences

**Next lesson:** We'll create a loop where LLM calls a tool, sees the result, and can call another tool if needed.